# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toBESkiii/FlyRank_AI_Intership/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



My confirmed lane is **Refresh / Content Opportunity Scoring**.

My baseline idea is that a page may deserve earlier review when it has not been updated recently but still receives meaningful search visibility.

In plain words:

> Prioritise pages that are stale but still visible in search.

Before encoding this rule, I will test the two signals it depends on:

1. **Staleness:** Do pages that have gone longer without an update show a higher observed decline rate?
2. **Search visibility:** How does the observed decline rate vary across impression-volume groups?

The observed decline proxy is used only to evaluate the signals. It will not be used as an input to calculate the baseline score.

After viewing the bucket tables, I will give each signal one verdict: **CONFIRMED, OPPOSITE, MIXED or FALSE**.

* Staleness verdict: **TO BE COMPLETED AFTER RUNNING THE TABLE**
* Search visibility verdict: **TO BE COMPLETED AFTER RUNNING THE TABLE**

If the signals are sufficiently supported, the rule will output one reason code:

`stale_but_visible`

The corresponding action label will be:

`REVIEW_FOR_REFRESH`


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "toBESkiii/FlyRank_AI_Intership/main/"
    "data/raw/content_refresh_anonymized.csv"
)

baseline_dataset = pd.read_csv(DATA_URL)

# This proxy is used only to evaluate the signals.
# It must not be used to calculate the baseline score.
baseline_dataset["decline_proxy"] = (
    baseline_dataset["trend_direction"] == "down"
).astype(int)

print("Dataset loaded successfully.")
print("Pages analysed:", len(baseline_dataset))


# ---------------------------------------------------------
# SIGNAL 1: STALENESS
# ---------------------------------------------------------

freshness_order = [
    "0-30",
    "31-90",
    "91-180",
    "181+",
    "never"
]

baseline_dataset["freshness_tier"] = pd.Categorical(
    baseline_dataset["freshness_tier"],
    categories=freshness_order,
    ordered=True
)

staleness_bucket_table = (
    baseline_dataset
    .groupby("freshness_tier", observed=True)
    .agg(
        n=("content_id", "count"),
        declining_pages=("decline_proxy", "sum"),
        decline_rate=("decline_proxy", "mean")
    )
    .reset_index()
)

staleness_bucket_table["decline_rate_pct"] = (
    staleness_bucket_table["decline_rate"] * 100
).round(1)

staleness_bucket_table = staleness_bucket_table[
    [
        "freshness_tier",
        "n",
        "declining_pages",
        "decline_rate_pct"
    ]
]

print("\nSignal 1 — Staleness bucket table")
display(staleness_bucket_table)


# ---------------------------------------------------------
# SIGNAL 2: SEARCH VISIBILITY
# ---------------------------------------------------------

impression_order = [
    "no_data",
    "none",
    "low",
    "moderate",
    "good",
    "excellent"
]

baseline_dataset["impression_tier"] = pd.Categorical(
    baseline_dataset["impression_tier"],
    categories=impression_order,
    ordered=True
)

visibility_bucket_table = (
    baseline_dataset
    .groupby("impression_tier", observed=True)
    .agg(
        n=("content_id", "count"),
        declining_pages=("decline_proxy", "sum"),
        decline_rate=("decline_proxy", "mean")
    )
    .reset_index()
)

visibility_bucket_table["decline_rate_pct"] = (
    visibility_bucket_table["decline_rate"] * 100
).round(1)

visibility_bucket_table = visibility_bucket_table[
    [
        "impression_tier",
        "n",
        "declining_pages",
        "decline_rate_pct"
    ]
]

print("\nSignal 2 — Search visibility bucket table")
display(visibility_bucket_table)


Dataset loaded successfully.
Pages analysed: 30000

Signal 1 — Staleness bucket table


,freshness_tier,n,declining_pages,decline_rate_pct
0,0-30,20480,10473,51.1
1,31-90,175,103,58.9
2,91-180,9171,5604,61.1
3,181+,174,82,47.1



Signal 2 — Search visibility bucket table


,impression_tier,n,declining_pages,decline_rate_pct
0,low,11248,5106,45.4
1,moderate,10469,6435,61.5
2,good,7205,4223,58.6
3,excellent,1078,498,46.2


### Signal verdicts

**Staleness verdict: MIXED**

The decline rate increased from 51.1% for pages updated within 30 days to 61.1% for pages last updated between 91 and 180 days. However, it fell to 47.1% for pages older than 180 days. Therefore, the data does not show that decline consistently increases as pages become older.

**Search visibility verdict: MIXED**

The moderate and good impression groups had relatively high decline rates of 61.5% and 58.6%. However, the excellent-impression group had a lower decline rate of 46.2%. Therefore, higher search visibility does not consistently mean that a page is more likely to decline.

Despite these mixed results, staleness and visibility can still be used as simple prioritisation signals. They should not be described as reliable predictors of decline.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


Based on the signal checks, I will use staleness and search visibility as simple prioritisation signals rather than claiming that either one independently predicts decline.

The baseline rule is:

> Prioritise a page for refresh review when it has not been updated for at least 90 days and has received at least 300 search impressions during the last 90 days.

A page must meet both conditions to enter the ranked review queue.

For qualifying pages, the baseline score is the page's 90-day impression count. This means pages with greater existing search visibility appear higher in the queue because a larger amount of search exposure may be affected.

The rule produces one reason code:

`stale_but_visible`

The action label is:

`REVIEW_FOR_REFRESH`

The score does not use `trend_direction`, `trend_pct`, the decline proxy, future information or page/client identifiers. The decline proxy is used only after scoring to evaluate the baseline.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

# ---------------------------------------------------------
# 1. Define the transparent baseline conditions
# ---------------------------------------------------------

minimum_days_since_update = 90
minimum_impressions = 300

stale_condition = (
    baseline_dataset["days_since_last_update"]
    >= minimum_days_since_update
)

visible_condition = (
    baseline_dataset["impressions_90d"]
    >= minimum_impressions
)

# A page qualifies only when both conditions are true
baseline_dataset["qualifies_for_review"] = (
    stale_condition & visible_condition
)

# ---------------------------------------------------------
# 2. Create the baseline score
# ---------------------------------------------------------

# Qualifying pages receive their impressions as the score.
# Non-qualifying pages receive zero.
baseline_dataset["baseline_score"] = (
    baseline_dataset["qualifies_for_review"].astype(int)
    * baseline_dataset["impressions_90d"]
)

# ---------------------------------------------------------
# 3. Attach one reason code and one action label
# ---------------------------------------------------------

baseline_dataset["reason_code"] = ""

baseline_dataset.loc[
    baseline_dataset["qualifies_for_review"],
    "reason_code"
] = "stale_but_visible"

baseline_dataset["action_label"] = ""

baseline_dataset.loc[
    baseline_dataset["qualifies_for_review"],
    "action_label"
] = "REVIEW_FOR_REFRESH"

# ---------------------------------------------------------
# 4. Build the ranked review queue
# ---------------------------------------------------------

queue_columns = [
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "freshness_tier",
    "impressions_90d",
    "impression_tier",
    "ctr",
    "avg_position",
    "decline_proxy"
]

ranked_review_queue = (
    baseline_dataset.loc[
        baseline_dataset["qualifies_for_review"],
        queue_columns
    ]
    .sort_values(
        by=["baseline_score", "days_since_last_update"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_review_queue.insert(
    0,
    "review_rank",
    range(1, len(ranked_review_queue) + 1)
)

# ---------------------------------------------------------
# 5. Evaluate the top of the queue
# ---------------------------------------------------------

top_k = min(10, len(ranked_review_queue))

precision_at_10 = (
    ranked_review_queue
    .head(top_k)["decline_proxy"]
    .mean()
)

base_decline_rate = baseline_dataset["decline_proxy"].mean()

print("Baseline rule:")
print(
    f"Days since last update >= {minimum_days_since_update}"
)
print(
    f"Impressions over 90 days >= {minimum_impressions:,}"
)

print("\nPages in the ranked queue:", len(ranked_review_queue))
print(f"Overall decline-proxy rate: {base_decline_rate:.3f}")
print(f"Baseline Precision@{top_k}: {precision_at_10:.3f}")

# ---------------------------------------------------------
# 6. Write the required CSV
# ---------------------------------------------------------

output_directory = Path("work/outputs")
output_directory.mkdir(parents=True, exist_ok=True)

output_path = (
    output_directory / "baseline_action_score.csv"
)

ranked_review_queue.to_csv(
    output_path,
    index=False
)

print("\nCSV written successfully:")
print(output_path)

# Display the ten highest-ranked pages
display(ranked_review_queue.head(10))


Baseline rule:
Days since last update >= 90
Impressions over 90 days >= 300

Pages in the ranked queue: 7234
Overall decline-proxy rate: 0.542
Baseline Precision@10: 0.600

CSV written successfully:
work/outputs/baseline_action_score.csv


,review_rank,content_id,client_id,baseline_score,reason_code,action_label,days_since_last_update,freshness_tier,impressions_90d,impression_tier,ctr,avg_position,decline_proxy
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,517715,excellent,0.14,4.2,1
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,443434,excellent,0.21,27.9,0
2,3,content_2c2606c5d176,client_19581e27de,347399,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,347399,excellent,0.53,4.2,1
3,4,content_cb112fce36be,client_19581e27de,309910,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,309910,excellent,0.16,5.6,1
4,5,content_9532f197bbc8,client_4e07408562,309192,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,309192,excellent,0.87,2.0,1
5,6,content_36ff89c8214e,client_19581e27de,295097,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,295097,excellent,0.05,7.3,0
6,7,content_b28d1efd668f,client_6208ef0f77,286608,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,286608,excellent,0.06,26.2,0
7,8,content_813e88069237,client_6208ef0f77,233561,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,233561,excellent,0.06,26.2,1
8,9,content_c21024970297,client_19581e27de,211366,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,211366,excellent,0.41,5.1,0
9,10,content_c8e9d6ab9013,client_19581e27de,208678,stale_but_visible,REVIEW_FOR_REFRESH,104,91-180,208678,excellent,0.00,9.7,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

##

I reviewed the ten highest-ranked pages produced by the baseline rule.

Every page received the action `REVIEW_FOR_REFRESH` because it had not been updated for at least 90 days and had received at least 300 impressions during the previous 90 days.

The review is intentionally sceptical. A high baseline score does not prove that a page needs changing. A recommendation may be wrong if the content remains accurate, the topic is seasonal, the page is intentionally evergreen, the search queries are not relevant to the business, or another factor explains its performance.

The code below prints one review line for each of the top ten pages, including the action, the reason it ranked and a possible reason why the recommendation could be wrong.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Select the ten highest-ranked pages
top_10_review = ranked_review_queue.head(10).copy()

# Calculate the median CTR for a simple comparison
median_ctr = baseline_dataset["ctr"].median()


def identify_possible_weakness(page_row):
    """
    Return one reasonable explanation for why the baseline
    recommendation could be wrong.
    """

    if page_row["decline_proxy"] == 0:
        return (
            "the page is not currently labelled as declining, "
            "so age and visibility may be creating a false positive"
        )

    if page_row["ctr"] >= median_ctr:
        return (
            "its CTR is already at or above the dataset median, "
            "so the search snippet may already be performing adequately"
        )

    if page_row["avg_position"] > 20:
        return (
            "its average search position is weak, so impressions alone "
            "may not represent a strong refresh opportunity"
        )

    if page_row["days_since_last_update"] >= 365:
        return (
            "the page may be intentionally evergreen and could still "
            "contain accurate information despite its age"
        )

    return (
        "the decline may be caused by seasonality, changing search demand "
        "or competition rather than outdated content"
    )


review_lines = []

for _, page_row in top_10_review.iterrows():
    possible_weakness = identify_possible_weakness(page_row)

    review_line = (
        f"Rank {int(page_row['review_rank'])}: "
        f"{page_row['action_label']}. "
        f"It ranked because it was last updated "
        f"{int(page_row['days_since_last_update'])} days ago and received "
        f"{int(page_row['impressions_90d']):,} impressions. "
        f"The recommendation could be wrong if {possible_weakness}."
    )

    review_lines.append(review_line)

print("TOP-10 BASELINE REVIEW\n")

for review_line in review_lines:
    print(review_line)
    print()


TOP-10 BASELINE REVIEW

Rank 1: REVIEW_FOR_REFRESH. It ranked because it was last updated 104 days ago and received 517,715 impressions. The recommendation could be wrong if its CTR is already at or above the dataset median, so the search snippet may already be performing adequately.

Rank 2: REVIEW_FOR_REFRESH. It ranked because it was last updated 104 days ago and received 443,434 impressions. The recommendation could be wrong if the page is not currently labelled as declining, so age and visibility may be creating a false positive.

Rank 3: REVIEW_FOR_REFRESH. It ranked because it was last updated 104 days ago and received 347,399 impressions. The recommendation could be wrong if its CTR is already at or above the dataset median, so the search snippet may already be performing adequately.

Rank 4: REVIEW_FOR_REFRESH. It ranked because it was last updated 104 days ago and received 309,910 impressions. The recommendation could be wrong if its CTR is already at or above the dataset med

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak recommendations and baseline limitations

The baseline rule is transparent, but it is intentionally simple and will produce some weak recommendations.

A page can rank highly because it is old and receives many impressions, even when its content remains accurate and useful. The rule also does not understand seasonality, search intent, business importance, content quality or recent changes in competition.

A particularly weak recommendation would be a page that ranks highly but is not labelled as declining. This suggests that staleness and visibility alone may have created a false positive.

Other recommendations may be questionable when the page already has a relatively strong click-through rate, is intentionally evergreen or receives impressions from searches that are not valuable to the business.

These weak picks demonstrate why this baseline should be treated as a comparison point rather than a final decision system. A future machine-learning model must improve the quality of the ranked queue while remaining understandable.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

# Review possible weaknesses within the top ten recommendations
weak_pick_review = top_10_review.copy()

weak_pick_review["possible_weakness"] = np.select(
    [
        weak_pick_review["decline_proxy"] == 0,
        weak_pick_review["ctr"] >= median_ctr,
        weak_pick_review["avg_position"] > 20,
        weak_pick_review["days_since_last_update"] >= 365
    ],
    [
        "Not labelled as declining — possible false positive",
        "CTR is already at or above the dataset median",
        "Weak average position may explain the opportunity differently",
        "May be intentionally evergreen despite its age"
    ],
    default=(
        "Performance may be affected by seasonality, demand "
        "or competition rather than stale content"
    )
)

# Prioritise the clearest weak picks first
weak_pick_review["clear_false_positive"] = (
    weak_pick_review["decline_proxy"] == 0
)

weak_pick_review = (
    weak_pick_review
    .sort_values(
        by=["clear_false_positive", "review_rank"],
        ascending=[False, True]
    )
)

print("Potential weak recommendations from the Top 10:")

display(
    weak_pick_review[
        [
            "review_rank",
            "content_id",
            "baseline_score",
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "avg_position",
            "decline_proxy",
            "possible_weakness"
        ]
    ]
)


Potential weak recommendations from the Top 10:


,review_rank,content_id,baseline_score,days_since_last_update,impressions_90d,ctr,avg_position,decline_proxy,possible_weakness
1,2,content_2dba2b1f9536,443434,104,443434,0.21,27.9,0,Not labelled as declining — possible false pos...
5,6,content_36ff89c8214e,295097,104,295097,0.05,7.3,0,Not labelled as declining — possible false pos...
6,7,content_b28d1efd668f,286608,104,286608,0.06,26.2,0,Not labelled as declining — possible false pos...
8,9,content_c21024970297,211366,104,211366,0.41,5.1,0,Not labelled as declining — possible false pos...
0,1,content_5fe46e04994d,517715,104,517715,0.14,4.2,1,CTR is already at or above the dataset median
2,3,content_2c2606c5d176,347399,104,347399,0.53,4.2,1,CTR is already at or above the dataset median
3,4,content_cb112fce36be,309910,104,309910,0.16,5.6,1,CTR is already at or above the dataset median
4,5,content_9532f197bbc8,309192,104,309192,0.87,2.0,1,CTR is already at or above the dataset median
7,8,content_813e88069237,233561,104,233561,0.06,26.2,1,Weak average position may explain the opportun...
9,10,content_c8e9d6ab9013,208678,104,208678,0.00,9.7,1,"Performance may be affected by seasonality, de..."


### Findings from the weak-pick review

The manual review identified four clear possible false positives in the top ten, at ranks 2, 6, 7 and 9. These pages received high baseline scores but were not labelled as declining.

Therefore, six of the top ten recommendations matched the decline proxy, giving the baseline a Precision@10 of 0.60.

All ten pages had been last updated 104 days earlier. This means that staleness did not distinguish the pages within the top of the queue, and the final ranking was driven mainly by their impression counts.

Several highly ranked pages also had strong CTR or strong average search positions. For example, some pages had CTR values above 0.40 or average positions close to the top of the search results. Refreshing these pages without further investigation could be unnecessary or even risky.

The review shows that the baseline is useful as a transparent starting point, but impressions and staleness alone are not enough to produce consistently reliable recommendations. A stronger model should distinguish between declining pages, healthy high-performing pages and pages affected by other factors such as seasonality or changing demand.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.